In [25]:
from random import random, randint, seed, sample
from icecream import ic
from math import sqrt
from collections import deque

In [26]:
NORD = 0
EST = 1
SUD = 2
OVEST = 3

DELTA_MOVIMENTO = {
    NORD: (0, 1),
    EST: (1, 0),
    SUD: (0, -1),
    OVEST: (-1, 0)
}

In [ ]:

from typing import Counter


class WumpusAgent:
    def __init__(self, grid_size=4):
        self.grid_size = grid_size
        
        # STATO DELL'AGENTE
        self.pos = (1, 1)          
        self.direzione = EST       
        self.ha_oro = False
        self.ha_freccia = True
        self.azioni_pianificate = deque()
        
        self.target_freccia = None 

        # MEMORIA DELL'AGENTE
        self.visitate = set()
        self.sicure = {(1, 1)}              
        self.sicure_non_visitate = set()    
        
        # Gestione Pozzi (Breeze)
        self.senza_pozzo = {(1, 1)}         
        self.pozzi_certi = set()           
        self.vincoli_pozzi = {}            
        
        # Gestione Wumpus (Stench)
        self.wumpus_vivo = True
        self.senza_wumpus = {(1, 1)}        
        self.sospetti_wumpus = None        
        self.pos_wumpus_certa = None        

    def choose_action(self, percezioni):

        if isinstance(percezioni, dict):
            stench = percezioni['stench']
            breeze = percezioni['breeze']
            glitter = percezioni['glitter']
            bump = percezioni['bump']
            scream = percezioni['scream']
        else:
            stench, breeze, glitter, bump, scream = percezioni

        # Gestisci conseguenze dell'azione di aver scoccato la freccia (se presente)
        if self.target_freccia is not None:
            self._handle_arrow_result(scream)

        # Aggiorna la conoscenza con le percezioni della cella attuale
        if self.pos not in self.visitate:
            self.visitate.add(self.pos)
            self.sicure_non_visitate.discard(self.pos)
            
            self._update_breeze_knowledge(self.pos, breeze)
            self._update_stench_knowledge(self.pos, stench)
            self._propagate_deductions()

        # PRIORITÀ 1: Se c'è il luccichio e non abbiamo l'oro, lo raccogliamo subito!
        if glitter and not self.ha_oro:
            self.ha_oro = True
            self.azioni_pianificate.clear()  
            piano_ritorno = self._bfs_to_targets([(1, 1)]) or []
            self.azioni_pianificate = deque(piano_ritorno + ['Climb'])
            return 'Grab'

        if self.ha_oro and not self.azioni_pianificate:
            piano_ritorno = self._bfs_to_targets([(1, 1)]) or []
            self.azioni_pianificate = deque(piano_ritorno + ['Climb'])
            

        # Se abbiamo già delle mosse in coda (calcolate da una BFS precedente), esegui la prossima
        if self.azioni_pianificate:
            azione = self.azioni_pianificate.popleft()
            self._update_internal_state(azione)
            return azione

        # PRIORITÀ 2: Esplora la cella sicura non visitata più vicina
        if self.sicure_non_visitate:
            self.azioni_pianificate.extend(deque(self._bfs_to_targets(self.sicure_non_visitate)) if self._bfs_to_targets(self.sicure_non_visitate) else deque())


        # PRIORITÀ 3: Bloccati! Se abbiamo la freccia e sospettiamo del Wumpus, andiamo a sparare
        if not self.azioni_pianificate and self.ha_freccia and self.wumpus_vivo and self.sospetti_wumpus:
            self._plan_wumpus_shot()

        # PRIORITÀ 4: Nessuna mossa sicura e niente Wumpus da uccidere -> Rischio calcolato o Ritirata
        if not self.azioni_pianificate:
            cella_meno_rischiosa = self._choose_least_risky_cell()
            if cella_meno_rischiosa is not None:
                piano = self._bfs_to_targets(
                    {cella_meno_rischiosa}, consenti_target_non_sicuro=True
                )
                if piano:
                    self.azioni_pianificate.extend(piano)
            else:
                piano_rientro = self._bfs_to_targets([(1, 1)])
                if piano_rientro:
                    self.azioni_pianificate.extend(piano_rientro)
                if self.pos == (1, 1) and not self.azioni_pianificate:
                    self.azioni_pianificate.append('Climb')

        # Esegui la prima mossa del nuovo piano appena generato
        azione = self.azioni_pianificate.popleft()
        self._update_internal_state(azione)
        return azione

    def _update_breeze_knowledge(self, cella, ha_breeze):
        adiacenti = self._adjacent_cells(cella)
        if not ha_breeze:
            for adiacente in adiacenti:
                if adiacente not in self.senza_pozzo:
                    self.senza_pozzo.add(adiacente)
        else:
            if cella not in self.vincoli_pozzi:
                self.vincoli_pozzi[cella] = set()
            for adiacente in adiacenti:
                if adiacente not in self.senza_pozzo:
                    self.vincoli_pozzi[cella].add(adiacente)

    def _update_stench_knowledge(self, cella, ha_stench):
        if not self.wumpus_vivo:
            return

        adiacenti = self._adjacent_cells(cella)
        if not ha_stench:
            for adiacente in adiacenti:
                if adiacente not in self.senza_wumpus:
                    self.senza_wumpus.add(adiacente)
            if self.sospetti_wumpus is not None:
                self.sospetti_wumpus -= self.senza_wumpus
        else:
            if self.sospetti_wumpus is None:
                self.sospetti_wumpus = set()
                for adiacente in adiacenti:
                    if adiacente not in self.senza_wumpus:
                        self.sospetti_wumpus.add(adiacente)
            else:
                self.sospetti_wumpus &= {adiacente for adiacente in adiacenti if adiacente not in self.senza_wumpus}

    def _propagate_deductions(self):
        for cella in list(self.vincoli_pozzi.keys()):
            self.vincoli_pozzi[cella] -= self.senza_pozzo
            if len(self.vincoli_pozzi[cella]) == 1:
                pozzo_certo = next(iter(self.vincoli_pozzi[cella]))
                self.pozzi_certi.add(pozzo_certo)
                del self.vincoli_pozzi[cella]

        if self.sospetti_wumpus is not None and len(self.sospetti_wumpus) == 1:
            self.pos_wumpus_certa = next(iter(self.sospetti_wumpus))
            for x in range(1, self.grid_size + 1):
                for y in range(1, self.grid_size + 1):
                    if (x, y) != self.pos_wumpus_certa:
                        self.senza_wumpus.add((x, y))

        self.sicure = self.senza_pozzo & self.senza_wumpus
        
        self.sicure_non_visitate = self.sicure - self.visitate

    def _handle_arrow_result(self, scream):
        if scream:
            self.wumpus_vivo = False
            self.senza_wumpus = {(x, y) for x in range(1, self.grid_size + 1) for y in range(1, self.grid_size + 1)}
        else:
            self.senza_wumpus.add(self.target_freccia)
            self.sospetti_wumpus.discard(self.target_freccia)
        self.target_freccia = None
        self._propagate_deductions()


    def _bfs_to_targets(self, celle_obiettivo, consenti_target_non_sicuro=False):
        coda = deque([(self.pos[0], self.pos[1], self.direzione, [])])
        visitati_bfs = set()
        visitati_bfs.add((self.pos[0], self.pos[1], self.direzione))

        target_coords = {(val[0], val[1]) for val in celle_obiettivo}
        target_stati = {val for val in celle_obiettivo if len(val) == 3}

        while coda:
            x, y, dir_attuale, azioni_fatte = coda.popleft()
            
            # Controlla se abbiamo raggiunto uno degli obiettivi (con o senza vincolo di direzione)
            if (x, y, dir_attuale) in target_stati or ((x, y) in target_coords and not target_stati):
                return azioni_fatte

            for azione in ['TurnLeft', 'TurnRight', 'Forward']:
                if azione == 'TurnLeft':
                    nuova_dir, nuova_pos = (dir_attuale - 1) % 4, (x, y)
                elif azione == 'TurnRight':
                    nuova_dir, nuova_pos = (dir_attuale + 1) % 4, (x, y)
                else:
                    dx, dy = DELTA_MOVIMENTO[dir_attuale]
                    nuova_pos, nuova_dir = (x + dx, y + dy), dir_attuale

                if 1 <= nuova_pos[0] <= self.grid_size and 1 <= nuova_pos[1] <= self.grid_size:
                    cella_ok = (nuova_pos in self.sicure) or (consenti_target_non_sicuro and nuova_pos in target_coords)
                    if cella_ok and (nuova_pos[0], nuova_pos[1], nuova_dir) not in visitati_bfs:
                        visitati_bfs.add((nuova_pos[0], nuova_pos[1], nuova_dir))
                        coda.append((nuova_pos[0], nuova_pos[1], nuova_dir, azioni_fatte + [azione]))
        return []
        

    def _plan_wumpus_shot(self):
        if (self.sospetti_wumpus is None or not self.sospetti_wumpus) and self.pos_wumpus_certa is None:
            return
        else:
            if self.pos_wumpus_certa is not None:
                wx, wy = self.pos_wumpus_certa
            else:
                wx, wy = next(iter(self.sospetti_wumpus))

            # Trova tutte le celle sicure allineate con (wx, wy)
            celle_tiro = []
            for x in range(1, self.grid_size + 1):
                if (x, wy) in self.sicure:
                    celle_tiro.append((x, wy))
            for y in range(1, self.grid_size + 1):
                if (wx, y) in self.sicure:
                    celle_tiro.append((wx, y))

            # Per ogni cella di tiro, calcola la direzione corretta per sparare verso (wx, wy)
            for cella_tiro in celle_tiro:
                x_tiro, y_tiro = cella_tiro
                if x_tiro == wx and y_tiro < wy:
                    dir_tiro = NORD
                elif x_tiro == wx and y_tiro > wy:
                    dir_tiro = SUD
                elif y_tiro == wy and x_tiro < wx:
                    dir_tiro = EST
                elif y_tiro == wy and x_tiro > wx:
                    dir_tiro = OVEST
                else:
                    continue

                cella_tiro_con_dir = (x_tiro, y_tiro, dir_tiro)

                azioni_per_raggiungere = self._bfs_to_targets({cella_tiro_con_dir}, consenti_target_non_sicuro=True)
                if azioni_per_raggiungere is not None:
                    self.azioni_pianificate.extend(azioni_per_raggiungere)
                    self.azioni_pianificate.append('Shoot')
                    self.target_freccia = (wx, wy)
                    break


    def _choose_least_risky_cell(self):
        frontiera = {adj for c in self.sicure for adj in self._adjacent_cells(c)}
        candidati = frontiera - self.visitate - self.pozzi_certi - {self.pos_wumpus_certa}
        if not candidati:
            return None

        rischi = Counter()
        for v in self.vincoli_pozzi.values():
            for c in v & candidati:
                rischi[c] += 1.0 / len(v)

        if self.wumpus_vivo and self.sospetti_wumpus:
            for c in self.sospetti_wumpus & candidati:
                rischi[c] += 1.0 / len(self.sospetti_wumpus)

        migliore = min(candidati, key=lambda c: rischi[c])
        return migliore if rischi[migliore] <= soglia else None


    def _adjacent_cells(self, cella):
        x, y = cella
        risultato = []
        for dx, dy in DELTA_MOVIMENTO.values():
            nx, ny = x + dx, y + dy
            if 1 <= nx <= self.grid_size and 1 <= ny <= self.grid_size:
                risultato.append((nx, ny))
        return risultato

    def _update_internal_state(self, azione):
        if azione == 'TurnRight':
            self.direzione = (self.direzione + 1) % 4
        elif azione == 'TurnLeft':
            self.direzione = (self.direzione - 1) % 4
        elif azione == 'Forward':
            dx, dy = DELTA_MOVIMENTO[self.direzione]
            self.pos = (self.pos[0] + dx, self.pos[1] + dy)
        elif azione == 'Shoot':
            self.ha_freccia = False

In [ ]:
class WumpusEnvironment:
    def __init__(self, grid_size=4, seed_value=None):
        self.grid_size = grid_size
        self.seed_value = seed_value
        if seed_value is not None:
            seed(seed_value)
        self._reset()

    def _reset(self):
        # Posizione iniziale dell'agente
        self.agent_pos = (1, 1)
        self.agent_dir = EST
        self.agent_has_gold = False
        self.agent_has_arrow = True
        self.wumpus_alive = True

        # Posizione casuale del Wumpus e dell'oro
        self.wumpus_pos = (randint(1, self.grid_size), randint(1, self.grid_size))
        while self.wumpus_pos == (1, 1):
            self.wumpus_pos = (randint(1, self.grid_size), randint(1, self.grid_size))

        self.gold_pos = (randint(1, self.grid_size), randint(1, self.grid_size))
        while self.gold_pos == (1, 1) or self.gold_pos == self.wumpus_pos:
            self.gold_pos = (randint(1, self.grid_size), randint(1, self.grid_size))

        # Posizioni dei pozzi
        num_pits = max(2, int(self.grid_size * 0.50))
        all_positions = [(x, y) for x in range(1, self.grid_size + 1) for y in range(1, self.grid_size + 1)]
        all_positions.remove((1, 1))
        all_positions.remove(self.wumpus_pos)
        all_positions.remove(self.gold_pos)
        pit_positions = sample(all_positions, num_pits)
        self.pit_positions = set(pit_positions)

    def get_perceptions(self):
        x, y = self.agent_pos
        stench = (x, y) in self._adjacent_cells(self.wumpus_pos)
        breeze = any((x, y) in self._adjacent_cells(pit) for pit in self.pit_positions)
        glitter = (x, y) == self.gold_pos

        return {
            'stench': stench,
            'breeze': breeze,
            'glitter': glitter,
            'bump': False,
            'scream': False
        }

    def _adjacent_cells(self, cella):
        x, y = cella
        return {(x + dx, y + dy) for dx, dy in DELTA_MOVIMENTO.values()
                if 1 <= x + dx <= self.grid_size and 1 <= y + dy <= self.grid_size}

    def _is_wumpus_hit(self):
        if not self.wumpus_alive:
            return False
        x, y = self.agent_pos
        dx, dy = DELTA_MOVIMENTO[self.agent_dir]
        # Controlla se il Wumpus è nella linea di tiro
        if dx != 0:
            if y == self.wumpus_pos[1] and ((dx > 0 and x < self.wumpus_pos[0]) or (dx < 0 and x > self.wumpus_pos[0])):
                self.wumpus_alive = False
                return True
        elif dy != 0: 
            if x == self.wumpus_pos[0] and ((dy > 0 and y < self.wumpus_pos[1]) or (dy < 0 and y > self.wumpus_pos[1])):
                self.wumpus_alive = False
                return True
        return False

    def execute_action(self, azione):
        x, y = self.agent_pos
        if azione == 'TurnRight':
            self.agent_dir = (self.agent_dir + 1) % 4
        elif azione == 'TurnLeft':
            self.agent_dir = (self.agent_dir - 1) % 4
        elif azione == 'Forward':
            dx, dy = DELTA_MOVIMENTO[self.agent_dir]
            new_x, new_y = x + dx, y + dy
            if 1 <= new_x <= self.grid_size and 1 <= new_y <= self.grid_size:
                self.agent_pos = (new_x, new_y)
            else:
                return {'bump': True}
        elif azione == 'Grab':
            if self.agent_pos == self.gold_pos:
                self.agent_has_gold = True
                self.gold_pos = None 
        elif azione == 'Shoot':
            if self.agent_has_arrow:
                self.agent_has_arrow = False
                if self._is_wumpus_hit():
                    return {'scream': True}
        elif azione == 'Climb':
            if self.agent_pos == (1, 1):
                return {'climb': True}

        return {'bump': False, 'scream': False, 'climb': False}

In [ ]:
# Game loop

def play_game(grid_size=4, seed_value=None, max_passi=100, verbose=True):
    env = WumpusEnvironment(grid_size=grid_size, seed_value=seed_value)
    agent = WumpusAgent(grid_size=grid_size)

    punteggio = 0
    passo = 0
    
    ultimo_esito = {'bump': False, 'scream': False, 'climb': False}

    if verbose:
        print("=== INIZIO PARTITA WUMPUS WORLD ===")
        print(f"Griglia: {grid_size}x{grid_size} | Wumpus: {env.wumpus_pos} | Oro: {env.gold_pos} | Pozzi: {env.pit_positions}\n")

    while passo < max_passi:
        passo += 1

        # Raccoglie le percezioni della cella attuale e integra bump/scream dall'azione precedente
        percezioni = env.get_perceptions()
        percezioni['bump'] = ultimo_esito['bump']
        percezioni['scream'] = ultimo_esito['scream']

        # L'agente decide l'azione in base alle percezioni
        azione = agent.choose_action(percezioni)

        if verbose:
            perc_attive = [k for k, v in percezioni.items() if v]
            print(f"Turno {passo:02d} | Pos: {env.agent_pos} ({env.agent_dir}) | "
                  f"Percezioni: {perc_attive} -> Azione: {azione}")

        # Esegue l'azione nell'ambiente
        ultimo_esito = env.execute_action(azione)

        # Aggiorna i costi dell'azione (-10 per Shoot, -1 per tutte le altre)
        punteggio += -10 if azione == 'Shoot' else -1

        # Controllo Uscita dalla caverna (Climb in (1,1))
        if ultimo_esito['climb']:
            if env.agent_has_gold:
                punteggio += 1000
                if verbose:
                    print(f"\n🏆 VITTORIA! L'agente è uscito dalla caverna con l'ORO! Punteggio finale: {punteggio}")
            else:
                if verbose:
                    print(f"\n🚪 RITIRATA! L'agente è uscito vivo ma senza oro. Punteggio finale: {punteggio}")
            break

        # Controllo Morte (Pozzo o Wumpus vivo)
        if env.agent_pos in env.pit_positions:
            punteggio -= 1000
            if verbose:
                print(f"\n💀 GAME OVER! L'agente è caduto in un pozzo in {env.agent_pos}. Punteggio finale: {punteggio}")
            break

        if env.agent_pos == env.wumpus_pos and env.wumpus_alive:
            punteggio -= 1000
            if verbose:
                print(f"\n💀 GAME OVER! L'agente è stato divorato dal Wumpus in {env.wumpus_pos}. Punteggio finale: {punteggio}")
            break
    else:
        if verbose:
            print(f"\n⏱️ TEMPO SCADUTO! Raggiunto il limite di {max_passi} passi. Punteggio: {punteggio}")

    return punteggio, env.agent_has_gold


if __name__ == "__main__":
    play_game(grid_size=4, seed_value=42, verbose=True)

=== INIZIO PARTITA WUMPUS WORLD ===
Griglia: 6x6 | Wumpus: (6, 1) | Oro: (1, 6) | Pozzi: {(3, 5), (4, 2), (3, 6)}

Turno 01 | Pos: (1, 1) (1) | Percezioni: [] -> Azione: Forward
Turno 02 | Pos: (2, 1) (1) | Percezioni: [] -> Azione: Forward
Turno 03 | Pos: (3, 1) (1) | Percezioni: [] -> Azione: Forward
Turno 04 | Pos: (4, 1) (1) | Percezioni: ['breeze'] -> Azione: TurnLeft
Turno 05 | Pos: (4, 1) (0) | Percezioni: ['breeze'] -> Azione: TurnLeft
Turno 06 | Pos: (4, 1) (3) | Percezioni: ['breeze'] -> Azione: Forward
Turno 07 | Pos: (3, 1) (3) | Percezioni: [] -> Azione: TurnRight
Turno 08 | Pos: (3, 1) (0) | Percezioni: [] -> Azione: Forward
Turno 09 | Pos: (3, 2) (0) | Percezioni: ['breeze'] -> Azione: TurnLeft
Turno 10 | Pos: (3, 2) (3) | Percezioni: ['breeze'] -> Azione: Forward
Turno 11 | Pos: (2, 2) (3) | Percezioni: [] -> Azione: Forward
Turno 12 | Pos: (1, 2) (3) | Percezioni: [] -> Azione: TurnRight
Turno 13 | Pos: (1, 2) (0) | Percezioni: [] -> Azione: Forward
Turno 14 | Pos: (1,